# Step 2 — Catalogue Cleaning and Deduplication

## Project: Skincare Product Catalogue

### Objective
Clean and standardize the product catalogue, identify potential
duplicates, review ingredient-list quality, and document manual
verification decisions.

### Dataset
- Input: `data/raw/moisturizers_with_cosing.csv`
- Total records: 244
- Original data is preserved.
- No records are removed automatically.

### Workflow
1. Setup and configuration
2. Initial data quality assessment
3. Standardize product information
4. Detect potential duplicates
5. Inspect ingredient-list quality
6. Load and apply manual-review decisions
7. Final validation and export

### Important Notes
- Manual ingredient and duplicate reviews are stored separately.
- Uncertain duplicate pairs are retained.
- Sunscreen review is postponed.
- Outputs remain provisional until final validation.

## 2.1 — Setup and load a protected working copy
Run cells **from top to bottom**. Install optional packages in the notebook kernel if needed: `%pip install pandas rapidfuzz openpyxl`.

In [1]:
# Standard libraries
import re
from pathlib import Path
from itertools import combinations

# Third-party libraries
import pandas as pd
from rapidfuzz import fuzz

from pathlib import Path

# Find the project root
current_dir = Path.cwd().resolve()

if (current_dir / "data" / "raw").exists():
    project_dir = current_dir

elif (current_dir.parent / "data" / "raw").exists():
    project_dir = current_dir.parent

else:
    raise FileNotFoundError(
        "Project root not found. "
        "Open VS Code from the skincare-project folder."
    )

# Define project data directories
raw_path = project_dir / "data" / "raw"
interim_path = project_dir / "data" / "interim"

# Create output directory if it doesn't exist
interim_path.mkdir(parents=True, exist_ok=True)

# Input data
input_file = raw_path / "moisturizers_with_cosing.csv"

if not input_file.is_file():
    raise FileNotFoundError(
        f"Input dataset not found: {input_file}"
    )

print("Project root:", project_dir)
print("Input dataset:", input_file)

Project root: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project
Input dataset: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/raw/moisturizers_with_cosing.csv


In [2]:
# Load the original dataset
df = pd.read_csv(
    input_file,
    dtype={"code": "string"}
)

# Create a working copy
df_clean = df.copy()

# Preserve original row positions for traceability
df_clean["source_row"] = range(len(df_clean))

# Display a short loading summary
print(f"Products loaded: {len(df)}")
print(f"Original columns: {len(df.columns)}")

display(df_clean.head(3))

Products loaded: 244
Original columns: 17


,role,code,product_name,brands,france,eu,ingredients_text,product_row_id,parsed_ingredient_count,matched_ingredient_count,ambiguous_ingredient_count,unmatched_ingredient_count,match_fraction,cosing_ids,matched_inci_names,cosing_functions,unmatched_ingredients,source_row
0,moisturizer,0860015354624,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,False,False,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,190,1,0,0,1,0.0,NaN,NaN,NaN,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,0
1,moisturizer,0860015354648,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,False,False,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,191,1,0,0,1,0.0,NaN,NaN,NaN,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,1
2,moisturizer,4070765004748,Gesichtscreme Niacinamide LSF 30,Balea,False,False,"AQUA, NIACINAMIDE, DIBUTYL ADIPATE, GLYCERIN, ...",192,25,25,0,0,1.0,31713 | 31959 | 32858 | 33893 | 33895 | 33896 ...,AQUA | BIS-ETHYLHEXYLOXYPHENOL METHOXYPHENYL T...,ANTIMICROBIAL | ANTISTATIC | BLEACHING | BUFFE...,NaN,2


## 2.2 — Baseline inspection
Check the real fields and missing data before edits. Counts in this notebook are calculated from your file, not hardcoded assumptions.

In [3]:
# ============================================================
# Initial Data Quality Assessment
# ============================================================

# 1. Dataset dimensions
print("DATASET OVERVIEW")
print(f"Products: {len(df_clean)}")
print(f"Columns: {len(df.columns)}")

# 2. Missing values in important fields
key_columns = [
    "code",
    "brands",
    "product_name",
    "ingredients_text"
]

print("\nMISSING VALUES")
missing_summary = (
    df_clean[key_columns]
    .isna()
    .sum()
    .to_frame(name="Missing values")
)

display(missing_summary)

# 3. Initial category distribution
print("\nPRODUCT CATEGORIES")
display(df_clean["role"].value_counts(dropna=False))

# 4. Preview the dataset
print("\nDATASET PREVIEW")
display(df_clean.head(5))

DATASET OVERVIEW
Products: 244
Columns: 17

MISSING VALUES


,Missing values
code,0
brands,2
product_name,0
ingredients_text,0



PRODUCT CATEGORIES


role
moisturizer    244
Name: count, dtype: int64


DATASET PREVIEW


,role,code,product_name,brands,france,eu,ingredients_text,product_row_id,parsed_ingredient_count,matched_ingredient_count,ambiguous_ingredient_count,unmatched_ingredient_count,match_fraction,cosing_ids,matched_inci_names,cosing_functions,unmatched_ingredients,source_row
0,moisturizer,0860015354624,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,False,False,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,190,1,0,0,1,0.0000,NaN,NaN,NaN,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,0
1,moisturizer,0860015354648,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,False,False,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,191,1,0,0,1,0.0000,NaN,NaN,NaN,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,1
2,moisturizer,4070765004748,Gesichtscreme Niacinamide LSF 30,Balea,False,False,"AQUA, NIACINAMIDE, DIBUTYL ADIPATE, GLYCERIN, ...",192,25,25,0,0,1.0000,31713 | 31959 | 32858 | 33893 | 33895 | 33896 ...,AQUA | BIS-ETHYLHEXYLOXYPHENOL METHOXYPHENYL T...,ANTIMICROBIAL | ANTISTATIC | BLEACHING | BUFFE...,NaN,2
3,moisturizer,8437018443420,RF Facial Cream,Masderm,False,True,"Aqua, Paraffinum Liquidum, Glyceryl Stearate, ...",193,33,30,0,3,0.9091,31959 | 32858 | 33401 | 34040 | 34103 | 34924 ...,AQUA | ASTROCARYUM MURUMURU SEED BUTTER | BIDE...,ANTICORROSIVE | ANTIFOAMING | ANTIMICROBIAL | ...,Citrus Aurantium Bergamia Oil | Tetramethyl Ac...,3
4,moisturizer,7891010255725,Hydro Boost,Neutrogena,False,False,"Aqua/água purificada, Dimethicone/dimeticona, ...",194,19,0,0,19,0.0000,NaN,NaN,NaN,Aqua/água purificada | Dimethicone/dimeticona ...,4


## 2.3 — Standardize Product Information

This section standardizes categories, brand names, and product names
while preserving the original data.

We create:
- `category_clean`: Standardized original category label.
- `brand_clean`: Brand name with extra whitespace removed.
- `product_name_clean`: Product name with extra whitespace removed.
- `brand_key`: Lowercase brand name for duplicate detection.
- `product_key`: Lowercase product name for duplicate detection.

**Important:** Category corrections identified during manual review
are handled separately. We do not overwrite the original `role`.

In [4]:
# ============================================================
# Standardize Product Information
# ============================================================

# 1. Standardize category labels
df_clean["category_clean"] = (
    df_clean["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# 2. Standardize brand and product names
for original, cleaned in [
    ("brands", "brand_clean"),
    ("product_name", "product_name_clean")
]:
    df_clean[cleaned] = (
        df_clean[original]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# 3. Create matching keys for duplicate detection
df_clean["brand_key"] = (
    df_clean["brand_clean"]
    .fillna("")
    .str.casefold()
)

df_clean["product_key"] = (
    df_clean["product_name_clean"]
    .fillna("")
    .str.casefold()
)

# 4. Display results
print("CATEGORY DISTRIBUTION")
display(df_clean["category_clean"].value_counts(dropna=False))

print("\nSTANDARDIZED PRODUCT PREVIEW")
display(
    df_clean[
        [
            "source_row",
            "brand_clean",
            "product_name_clean",
            "category_clean"
        ]
    ].head(10)
)

CATEGORY DISTRIBUTION


category_clean
moisturizer    244
Name: count, dtype: Int64


STANDARDIZED PRODUCT PREVIEW


,source_row,brand_clean,product_name_clean,category_clean
0,0,PLU Laboratories,PLU Laboratories GHK-Cu Face Tonic,moisturizer
1,1,PLU Laboratories,PLU Laboratories GHK-Cu Face Tonic,moisturizer
2,2,Balea,Gesichtscreme Niacinamide LSF 30,moisturizer
3,3,Masderm,RF Facial Cream,moisturizer
4,4,Neutrogena,Hydro Boost,moisturizer
5,5,Byoma,Hydrating recovery oil,moisturizer
6,6,Stanhome,All Purpose Crema Multiuso Profumo di Fico,moisturizer
7,7,Byphasse,Crème ultra-hydratante,moisturizer
8,8,Diadermine,ΕΝΥΔΑΤΙΚΗ ΚΡΕΜΑ ΣΕ ΜΟΡΦΗ ΤΖΕΛ,moisturizer
9,9,Ceelini,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,moisturizer


## 2.4 — Detect Potential Duplicate Products

This section identifies three types of possible duplicates:

1. **Exact duplicates:** Records with identical values across all original columns.
2. **Repeated barcodes:** Products sharing the same barcode.
3. **Similar product names:** Products from the same brand whose names have
   at least 90% similarity, measured using RapidFuzz.

Important:
- Similar names do not necessarily indicate duplicate products.
- Different sizes or formulations may have different barcodes.
- No products are automatically deleted.
- Final decisions are documented in the manual-review files.

In [5]:
# ============================================================
# Detect Potential Duplicate Products
# ============================================================

# 1. Exact duplicates
# Use the original dataset, not df_clean, because
# source_row makes each cleaned record unique.

exact_duplicates = df[
    df.duplicated(keep=False)
].copy()


# 2. Repeated product barcodes

repeated_codes = df_clean[
    df_clean["code"].notna()
    & df_clean["code"].duplicated(keep=False)
].copy()


# 3. Similar product names within the same brand

SIMILARITY_THRESHOLD = 90

candidates = []

for brand, group in df_clean.groupby("brand_key"):

    if not brand:
        continue

    for i, j in combinations(group.index, 2):

        name_1 = df_clean.loc[i, "product_key"]
        name_2 = df_clean.loc[j, "product_key"]

        if not name_1 or not name_2:
            continue

        similarity = fuzz.ratio(name_1, name_2)

        if similarity >= SIMILARITY_THRESHOLD:
            candidates.append({
                "row_1": int(df_clean.loc[i, "source_row"]),
                "row_2": int(df_clean.loc[j, "source_row"]),
                "brand": brand,
                "product_1": df_clean.loc[i, "product_name"],
                "product_2": df_clean.loc[j, "product_name"],
                "name_similarity": round(similarity, 2)
            })


possible_duplicates_df = pd.DataFrame(
    candidates,
    columns=[
        "row_1",
        "row_2",
        "brand",
        "product_1",
        "product_2",
        "name_similarity"
    ]
)


# 4. Summary

print("DUPLICATE DETECTION SUMMARY")
print("-" * 40)

print("Exact duplicate records:", len(exact_duplicates))
print("Repeated barcode records:", len(repeated_codes))
print("Similar-name candidate pairs:", len(possible_duplicates_df))

print("\nSIMILAR-NAME CANDIDATES")
display(possible_duplicates_df)

DUPLICATE DETECTION SUMMARY
----------------------------------------
Exact duplicate records: 0
Repeated barcode records: 0
Similar-name candidate pairs: 8

SIMILAR-NAME CANDIDATES


,row_1,row_2,brand,product_1,product_2,name_similarity
0,175,176,babylove,leichte Gesichtscreme,leichte Gesichtscreme,100.00
1,49,56,bioderma,Bioderma Atoderm Cream Ultra,Bioderma Atoderm Creme Ultra,96.43
2,85,136,bioderma,Sébium sensitive,Sébium sensitive,100.00
3,78,191,kiehl's,Ultra Facial Cream,Ultra facial cream,100.00
4,108,186,l'oréal,Revitalift filler,Revitalift filler,100.00
5,32,84,nivea,Nivea creme,Nivea Creme,100.00
6,0,1,plu laboratories,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories GHK-Cu Face Tonic,100.00
7,62,63,the inkey list,"Vitamin B, C and E moisturizer","Vitamin B, C and E moisturizer",100.00


## 2.5 — Inspect Ingredient Lists and Compare Formulations

This section checks ingredient-list formatting and compares
ingredient compositions between potential duplicate products.

We use one conservative parsing function to:
- Split comma-separated ingredient lists.
- Handle bullets and line breaks.
- Identify missing or incomplete ingredient lists.
- Flag known OCR and formatting problems for manual review.

The parsed ingredient lists are used only for preliminary
comparison. They do not replace the original ingredient text.

**Important:** Unverified ingredient lists are not treated
as confirmed formulations.

In [6]:
# ============================================================
# Ingredient Parsing and Quality Assessment
# ============================================================

# Records known to contain irregular ingredient formatting
MANUAL_REVIEW_ROWS = {39, 95, 99, 108, 114, 190}

# Records with verified period-separated ingredient formatting
PERIOD_SEPARATED_ROWS = {0, 1}

# Source entries that do not contain ingredient lists
NON_INGREDIENT_TEXT = {
    "ingredienti non visibili nell'immagine.",
    "günlük nemlendirici krem"
}


def parse_ingredients(text, source_row):
    """
    Parse ingredient text for preliminary quality assessment.

    Returns:
        ingredients: Alphabetically sorted ingredient strings.
        status: Parsing/review status.

    Note:
        This function does not verify INCI names or formulations.
    """

    # Missing ingredient text
    if pd.isna(text) or not str(text).strip():
        return [], "missing_ingredient_list"

    raw = str(text).strip()

    # Text that is a description rather than an ingredient list
    if raw.casefold() in NON_INGREDIENT_TEXT:
        return [], "missing_ingredient_list"

    # Standardize separators
    raw = re.sub(
        r"\[/\]\(https://file\+\.vscode-resource\.vscode-cdn\.net/\)",
        " / ",
        raw
    )

    raw = re.sub(r"[•·⚫]", ",", raw)
    raw = re.sub(r"\r?\n+", ",", raw)

    # Handle the known period-separated PLU ingredient lists
    if source_row in PERIOD_SEPARATED_ROWS:
        raw = re.sub(r"\.\s+", ", ", raw)

    # Split and standardize ingredient strings
    ingredients = [
        item.strip().casefold().strip(" .;")
        for item in raw.split(",")
        if item.strip(" .;")
    ]

    # Assign preliminary quality status
    status = (
        "parsed_candidate"
        if len(ingredients) >= 2
        else "needs_manual_review"
    )

    # Known problematic records require manual verification
    if source_row in MANUAL_REVIEW_ROWS:
        status = "needs_manual_review"

    return sorted(ingredients), status


# Apply the parser to every product
parsed_results = [
    parse_ingredients(text, int(source_row))
    for text, source_row in zip(
        df_clean["ingredients_text"],
        df_clean["source_row"]
    )
]

df_clean["ingredients_sorted"] = [
    result[0] for result in parsed_results
]

df_clean["ingredient_review_status"] = [
    result[1] for result in parsed_results
]


# Display a short quality summary
print("INGREDIENT PARSING SUMMARY")

display(
    df_clean["ingredient_review_status"]
    .value_counts(dropna=False)
    .to_frame("Number of products")
)

INGREDIENT PARSING SUMMARY


,Number of products
ingredient_review_status,
parsed_candidate,233
needs_manual_review,9
missing_ingredient_list,2


In [7]:
print("Total parsed products:", len(df_clean))

print("\nProducts requiring review:")
print(
    df_clean["ingredient_review_status"]
    .value_counts()
)

print("\nParser check:")
print(
    parse_ingredients(
        "Aqua, Glycerin, Panthenol",
        source_row=999
    )
)

Total parsed products: 244

Products requiring review:
ingredient_review_status
parsed_candidate           233
needs_manual_review          9
missing_ingredient_list      2
Name: count, dtype: int64

Parser check:
(['aqua', 'glycerin', 'panthenol'], 'parsed_candidate')


### 2.6 — Compare Ingredient Lists Between Potential Duplicates

This section compares the parsed ingredient lists of products
identified as potential duplicates.

We calculate Jaccard similarity:

    Similarity = Common ingredients / Unique ingredients × 100

Important:
- Similarity is based on ingredient names, not concentrations.
- Ingredient order is not considered.
- Similarity is calculated only when both lists are usable.
- Products are never merged automatically.
- Final duplicate decisions are stored separately.

In [8]:
# ============================================================
# Compare Ingredient Lists Between Potential Duplicates
# ============================================================

comparison_results = []

for _, candidate in possible_duplicates_df.iterrows():

    row_1 = int(candidate["row_1"])
    row_2 = int(candidate["row_2"])

    product_1 = df_clean.loc[
        df_clean["source_row"] == row_1
    ].iloc[0]

    product_2 = df_clean.loc[
        df_clean["source_row"] == row_2
    ].iloc[0]

    # Use the parsed ingredients from the previous section
    ingredients_1 = set(product_1["ingredients_sorted"])
    ingredients_2 = set(product_2["ingredients_sorted"])

    # Calculate similarity only for usable ingredient lists
    valid = (
        product_1["ingredient_review_status"] == "parsed_candidate"
        and product_2["ingredient_review_status"] == "parsed_candidate"
        and bool(ingredients_1)
        and bool(ingredients_2)
    )

    common = ingredients_1 & ingredients_2
    all_ingredients = ingredients_1 | ingredients_2

    similarity = (
        round(100 * len(common) / len(all_ingredients), 2)
        if valid else pd.NA
    )

    comparison_results.append({
        "row_1": row_1,
        "row_2": row_2,
        "brand": candidate["brand"],
        "ingredients_1": len(ingredients_1),
        "ingredients_2": len(ingredients_2),
        "common_ingredients": len(common) if valid else pd.NA,
        "ingredient_similarity_pct": similarity,
        "comparison_status": (
            "provisional_comparison"
            if valid else "unverified_ingredient_list"
        ),
        "only_row_1": (
            "; ".join(sorted(ingredients_1 - ingredients_2))
            if valid else ""
        ),
        "only_row_2": (
            "; ".join(sorted(ingredients_2 - ingredients_1))
            if valid else ""
        )
    })

ingredient_comparison = pd.DataFrame(comparison_results)

print("INGREDIENT SIMILARITY BETWEEN CANDIDATE PAIRS")

display(
    ingredient_comparison[
        [
            "row_1",
            "row_2",
            "brand",
            "ingredient_similarity_pct",
            "comparison_status"
        ]
    ]
)

INGREDIENT SIMILARITY BETWEEN CANDIDATE PAIRS


,row_1,row_2,brand,ingredient_similarity_pct,comparison_status
0,175,176,babylove,69.23,provisional_comparison
1,49,56,bioderma,100.0,provisional_comparison
2,85,136,bioderma,100.0,provisional_comparison
3,78,191,kiehl's,35.29,provisional_comparison
4,108,186,l'oréal,<NA>,unverified_ingredient_list
5,32,84,nivea,56.52,provisional_comparison
6,0,1,plu laboratories,100.0,provisional_comparison
7,62,63,the inkey list,88.0,provisional_comparison


In [9]:
print("Candidate pairs:", len(possible_duplicates_df))
print("Compared pairs:", len(ingredient_comparison))

assert len(ingredient_comparison) == len(possible_duplicates_df)

print("Ingredient comparison completed successfully!")

Candidate pairs: 8
Compared pairs: 8
Ingredient comparison completed successfully!


## 2.7 — Manual Review Results

This section loads previously completed manual-review decisions.

The detailed product investigations are preserved in
`02_manual_review_archive.ipynb`.

Three files contain the review results:

- `manual_ingredient_review.xlsx`: Ingredient corrections and verification notes.
- `additional_single_ingredient_review.csv`: Single-ingredient product checks.
- `duplicate_manual_review.csv`: Decisions on potential duplicate pairs.

The original product records are preserved.

Unverified formulations and unresolved duplicates remain flagged.

In [10]:
# ============================================================
# Load Completed Manual Review Results
# ============================================================

# Check that all required manual-review files exist

required_review_files = [
    interim_path / "manual_ingredient_review.xlsx",
    interim_path / "additional_single_ingredient_review.csv",
    interim_path / "duplicate_manual_review.csv"
]

missing_files = [
    path for path in required_review_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required manual-review files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )

print("All required manual-review files found.")

ingredient_review = pd.read_excel(
    interim_path / "manual_ingredient_review.xlsx",
    dtype={"code": "string"}
)

single_review = pd.read_csv(
    interim_path / "additional_single_ingredient_review.csv",
    dtype={"code": "string"}
)

duplicate_review = pd.read_csv(
    interim_path / "duplicate_manual_review.csv",
    dtype={"code_1": "string", "code_2": "string"}
)

print("Manual review files loaded successfully.")

All required manual-review files found.
Manual review files loaded successfully.


In [11]:
# ============================================================
# Manual Review Summary
# ============================================================

print("INGREDIENT REVIEWS")
display(
    ingredient_review["review_decision"]
    .value_counts(dropna=False)
    .to_frame("Products")
)

print("\nSINGLE-INGREDIENT REVIEWS")
display(
    single_review["review_decision"]
    .value_counts(dropna=False)
    .to_frame("Products")
)

print("\nDUPLICATE-PAIR DECISIONS")
display(
    duplicate_review["review_decision"]
    .value_counts(dropna=False)
    .to_frame("Pairs")
)

INGREDIENT REVIEWS


,Products
review_decision,
verified_retailer,5
unavailable,1
verified_manufacturer,1
verified_secondary_source,1



SINGLE-INGREDIENT REVIEWS


,Products
review_decision,
active_only,1
plausible_single_ingredient,1
formulation_unconfirmed,1



DUPLICATE-PAIR DECISIONS


,Pairs
review_decision,
keep_separate,5
needs_review,2
merge_as_duplicate,1


In [12]:
print("Catalogue products:", len(df_clean))
print("Duplicate candidates:", len(possible_duplicates_df))
print("Ingredient reviews:", len(ingredient_review))
print("Single-ingredient reviews:", len(single_review))
print("Duplicate reviews:", len(duplicate_review))

Catalogue products: 244
Duplicate candidates: 8
Ingredient reviews: 8
Single-ingredient reviews: 3
Duplicate reviews: 8


## 2.8 — Apply Reviewed Category Corrections

Some products were incorrectly classified as moisturizers
in the original dataset.

This section records the category corrections identified
during manual review.

The original `role` column is preserved for traceability.

In [13]:
# ============================================================
# Apply Reviewed Category Corrections
# ============================================================

category_corrections = {
    0: "serum_treatment",
    1: "serum_treatment",
    39: "cleanser",
    100: "acne_treatment",
    146: "face_body_oil",
    186: "serum_treatment",
    226: "face_body_oil"
}

# Preserve original categories
df_clean["category_original"] = df_clean["role"]

# Apply reviewed corrections
df_clean["category_clean"] = (
    df_clean["source_row"]
    .map(category_corrections)
    .fillna(df_clean["category_clean"])
)

# Display updated category distribution
print("UPDATED PRODUCT CATEGORIES")
display(
    df_clean["category_clean"]
    .value_counts()
    .to_frame("Products")
)

print("\nRECLASSIFIED PRODUCTS")
display(
    df_clean.loc[
        df_clean["source_row"].isin(category_corrections),
        [
            "source_row",
            "product_name",
            "category_original",
            "category_clean"
        ]
    ]
)

UPDATED PRODUCT CATEGORIES


,Products
category_clean,
moisturizer,237
serum_treatment,3
face_body_oil,2
cleanser,1
acne_treatment,1



RECLASSIFIED PRODUCTS


,source_row,product_name,category_original,category_clean
0,0,PLU Laboratories GHK-Cu Face Tonic,moisturizer,serum_treatment
1,1,PLU Laboratories GHK-Cu Face Tonic,moisturizer,serum_treatment
39,39,Hautklar,moisturizer,cleanser
100,100,Oxy 5,moisturizer,acne_treatment
146,146,huile de jojoba vierge,moisturizer,face_body_oil
186,186,Revitalift filler,moisturizer,serum_treatment
226,226,Huile de jojoba,moisturizer,face_body_oil


## 2.9 — Final Data Validation

This section validates the cleaned catalogue before export.

Checks include:
- Product count and unique source identifiers
- Missing essential product information
- Category distribution
- Manual-review coverage and unresolved duplicate pairs

The catalogue remains provisional because some ingredient
formulations and duplicate identities are still unverified.

In [14]:
# ============================================================
# Final Data Validation
# ============================================================

print("FINAL DATA VALIDATION")
print("=" * 50)

# 1. Check that no product records were accidentally lost
assert len(df_clean) == len(df), "Product count has changed!"

print(f"Products retained: {len(df_clean)}")

# 2. Check that source rows remain unique
assert df_clean["source_row"].is_unique, (
    "Duplicate source_row identifiers detected!"
)

print("Source row identifiers: Unique")

# 3. Check essential fields
essential_columns = [
    "code",
    "brands",
    "product_name",
    "ingredients_text"
]

missing_values = df_clean[essential_columns].isna().sum()

print("\nMISSING ESSENTIAL VALUES")
display(missing_values.to_frame("Missing values"))

# 4. Check cleaned categories
print("\nFINAL CATEGORY DISTRIBUTION")
display(
    df_clean["category_clean"]
    .value_counts(dropna=False)
    .to_frame("Products")
)

assert df_clean["category_clean"].notna().all(), (
    "Some products have no cleaned category!"
)

# 5. Check manual-review coverage
assert len(ingredient_review) == 8, (
    "Unexpected ingredient-review count!"
)

assert len(single_review) == 3, (
    "Unexpected single-ingredient review count!"
)

assert len(duplicate_review) == 8, (
    "Unexpected duplicate-review count!"
)

print("\nMANUAL REVIEW COVERAGE")
print(f"Ingredient reviews: {len(ingredient_review)}")
print(f"Single-ingredient reviews: {len(single_review)}")
print(f"Duplicate pairs reviewed: {len(duplicate_review)}")

# 6. Check unresolved duplicate pairs
unresolved_pairs = duplicate_review[
    duplicate_review["review_decision"] == "needs_review"
]

print("\nUNRESOLVED DUPLICATE PAIRS")
display(
    unresolved_pairs[
        ["row_1", "row_2", "review_decision"]
    ]
)

print("\nValidation completed successfully!")

FINAL DATA VALIDATION
Products retained: 244
Source row identifiers: Unique

MISSING ESSENTIAL VALUES


,Missing values
code,0
brands,2
product_name,0
ingredients_text,0



FINAL CATEGORY DISTRIBUTION


,Products
category_clean,
moisturizer,237
serum_treatment,3
face_body_oil,2
cleanser,1
acne_treatment,1



MANUAL REVIEW COVERAGE
Ingredient reviews: 8
Single-ingredient reviews: 3
Duplicate pairs reviewed: 8

UNRESOLVED DUPLICATE PAIRS


,row_1,row_2,review_decision
2,85,136,needs_review
6,0,1,needs_review



Validation completed successfully!


## 2.10 — Export Provisional Cleaned Catalogue

This section exports the cleaned catalogue and ingredient
comparison results.

The original dataset remains unchanged.

**Important:**
- All 244 product records are retained.
- Reviewed category corrections are included.
- Manual-review decisions remain in separate files.
- The proposed duplicate merge has not been applied.
- Ingredient corrections from the manual-review workbook
  have not yet been integrated into the catalogue.

These outputs are provisional and will be finalized later.

In [15]:
# ============================================================
# Export Provisional Catalogue
# ============================================================

# Define output files
catalogue_file = interim_path / "catalogue_provisional.csv"
comparison_file = interim_path / "ingredient_comparison.csv"

# Save the cleaned catalogue
df_clean.to_csv(
    catalogue_file,
    index=False
)

# Save ingredient comparison results
ingredient_comparison.to_csv(
    comparison_file,
    index=False
)

# Display export summary
print("EXPORT COMPLETED")
print("=" * 45)

print(f"Catalogue: {catalogue_file}")
print(f"Products exported: {len(df_clean)}")

print(f"\nIngredient comparisons: {comparison_file}")
print(f"Candidate pairs exported: {len(ingredient_comparison)}")

EXPORT COMPLETED
Catalogue: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/interim/catalogue_provisional.csv
Products exported: 244

Ingredient comparisons: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/interim/ingredient_comparison.csv
Candidate pairs exported: 8
